# M2 one-step Haken coarsening

This offline notebook drives the same `run` CLI contract with `coarsening.enabled: true`. **Observed** cells below report only measured artifacts: partition/compression tables, slow-subspace distance, slow-eigenvalue error, and lifted trajectory distortion for the two first-evidence merge methods.

**Interpretation caveats:** these are candidate slow-mode coordinates, not proven order parameters; the embedding uses no ConceptNet labels or relation text; distortion is measured against the fine full-system CPU float64 reference and is distinct from M1 rank-r reconstruction error; a small distortion value on this tiny fixture is not evidence of ConceptNet-wide plateaus. Baselines, null models, and the multiscale hierarchy are later increments.

In [ ]:
from pathlib import Path
import json
import tempfile

from semmap_haken.cli import main
from semmap_haken.notebook import resolve_notebook_paths, run_resource_preflight

ALLOW_PRODUCTION_DOWNLOAD = False
# M2 evidence uses the trusted strict CPU float64 reference path per the delivery plan.
EXECUTION = {'backend': 'cpu', 'device': 0, 'dtype': 'float64', 'workers': 1, 'reserved_cpu_cores': 1, 'threads_per_worker': 1, 'batch_size': 'auto', 'deterministic': True, 'allow_auto_fallback': True}
work = Path(tempfile.mkdtemp(prefix='semmap-haken-m2-'))
# A local 12-node ring keeps this notebook offline while satisfying the M1
# iterative-solver contract; it is not a scientific ConceptNet result.
fixture = work / 'm2_ring.tsv'
records = []
for index in range(12):
    neighbour = (index + 1) % 12
    for left, right in ((index, neighbour), (neighbour, index)):
        records.append(
            f'/a/r/edge\t/r/RelatedTo\t/c/en/n{left:02d}\t/c/en/n{right:02d}\t'
            + json.dumps({'weight': 1.0, 'dataset': 'notebook_fixture', 'sources': [], 'license': 'fixture'})
        )
fixture.write_text('\n'.join(records) + '\n', encoding='utf-8')
profile = work / 'profiles.yaml'
profile.write_text('name: smoke\nexpected_max_nodes: 1000\n', encoding='utf-8')
config = work / 'experiment.yaml'
config.write_text(f'''
paths: {{workspace_root: {work}, data_root: data, cache_root: cache, runs_root: runs}}
dataset: {{source: fixture, path: {fixture}, language: en, relations: [RelatedTo, IsA], min_weight: 1.0, max_nodes: 1000, component: largest}}
graph: {{directed: false, weight_transform: raw, operator: normalized_adjacency}}
runtime: {{profile: smoke, random_seed: 1729, resource_profile: {profile}}}
execution: {json.dumps(EXECUTION)}
dynamics: {{model: linear, alpha: 1.0, beta: 0.5, time_start: 0.0, time_stop: 2.0, time_steps: 9, perturbations_per_kind: 1, perturbation_seed: 1729, random_sparse_fraction: 0.25, storage_policy: all, max_storage_mb: 32}}
spectral: {{top_k: 5, max_r: 3}}
coarsening: {{enabled: true, methods: [connectivity_matching, unconstrained_matching], target_reduction: 0.4, embedding_weighting: none, seed: 1729, tie_breaking: distance_then_node_index, aggregation: sum, distance_threshold: null}}
''', encoding='utf-8')
paths = resolve_notebook_paths(config)
preflight = run_resource_preflight(None, paths.runs_root)
print({'offline': not ALLOW_PRODUCTION_DOWNLOAD, 'preflight_ok': preflight.ok, 'cpu_reference': True, 'workspace': str(work)})

In [ ]:
# Observed: prepare the sparse fixture graph, then run M1 + M2 through the same CLI contract.
assert main(['prepare', '--config', str(config)]) == 0
prepared = sorted((work / 'runs').glob('prepare-*'))[-1]
text = config.read_text(encoding='utf-8').replace('spectral: {top_k: 5, max_r: 3}', f'spectral: {{prepared_graph_dir: {prepared}, top_k: 5, max_r: 3}}')
config.write_text(text, encoding='utf-8')
assert main(['run', '--config', str(config)]) == 0
run_dir = sorted((work / 'runs').glob('run-*'))[-1]
manifest = json.loads((run_dir / 'manifest.json').read_text(encoding='utf-8'))
assert manifest['stages'] == {'spectral': 'completed', 'dynamics': 'completed', 'coarsening': 'completed'}
print({'run_dir': str(run_dir), 'stages': manifest['stages'], 'seeds': manifest['random_seeds'], 'artifact_count': len(manifest['artifacts'])})

In [ ]:
# Observed: per-method partition/compression and lifted distortion tables (modest derived data only).
rows = []
for method in ('connectivity_matching', 'unconstrained_matching'):
    metrics = json.loads((run_dir / 'coarsening' / method / 'metrics.json').read_text(encoding='utf-8'))
    result = metrics['result']
    mapping = json.loads((run_dir / 'coarsening' / method / 'mapping.json').read_text(encoding='utf-8'))
    rows.append({
        'method': method,
        'fine_nodes': result['fine_node_count'],
        'coarse_nodes': result['coarse_node_count'],
        'compression_ratio': round(result['compression_ratio'], 3),
        'merges_achieved_over_requested': f"{result['achieved_merges']}/{result['requested_merges']}",
        'shortfall': result['shortfall_reason'],
        'subspace_distance': result['subspace_projection_distance'],
        'slow_eigenvalue_error': result['slow_eigenvalue_max_abs_error'],
        'mean_trajectory_error': result['mean_trajectory_relative_error'],
        'coarse_spectral_method': result['coarse_spectral_method'],
        'cpu_reference_enforced': metrics['execution']['cpu_reference_enforced'],
        'supernode_sizes': mapping['supernode_sizes'],
    })
try:
    import pandas as pd
    display(pd.DataFrame(rows))
except ImportError:
    print(json.dumps(rows, indent=2, sort_keys=True))
merged_uris = {method: json.loads((run_dir / 'coarsening' / method / 'mapping.json').read_text(encoding='utf-8'))['parent_child'] for method in ('connectivity_matching', 'unconstrained_matching')}
print({'merge_membership_preserves_original_uris': all(all(uri.startswith('/c/en/') for uri in members) for mapping in merged_uris.values() for members in mapping.values())})

In [ ]:
# Observed: embedding metadata and per-perturbation trajectory summaries from the persisted artifacts.
for method in ('connectivity_matching', 'unconstrained_matching'):
    embedding = json.loads((run_dir / 'coarsening' / method / 'embedding_metadata.json').read_text(encoding='utf-8'))
    metrics = json.loads((run_dir / 'coarsening' / method / 'metrics.json').read_text(encoding='utf-8'))
    print({'method': method, 'selected_mode_indices': embedding['selected_mode_indices'], 'embedding_shape': embedding['shape'], 'weighting': embedding['weighting'], 'trajectory_relative_errors': metrics['result']['trajectory_relative_errors'], 'runtime_seconds': metrics['result']['runtime_seconds'], 'caveats': embedding['caveats']})